In [1]:
from pathlib import Path
import json

In [2]:
base_dir = Path.cwd()
data_dir = base_dir / "data"

raw_audio_dir = data_dir / "Fuquay_2-19_raw_audio"
processed_audio_dir = data_dir / "Processed Fuquay_2-19_audio"
processed_audio_dir.mkdir(parents = True, exist_ok = True)

In [3]:
raw_audio_files = [f for f in raw_audio_dir.iterdir()
               if f.is_file() and not f.name.startswith(".")]

In [4]:
#Import Methods from utils.py
from utils import prepare_audio_for_embedding, decode_audio, preprocess_audio, save_audio

/opt/anaconda3/envs/voice_id_env/lib/python3.11/site-packages/pydub/utils.py:170: RuntimeWarning: Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work
  warn("Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work", RuntimeWarning)


In [5]:
target_files = [
 Path('/Users/anshulchiranth/Desktop/Strike/Voice Experiments/Transcription/data/Fuquay_2-19_raw_audio/21-03-41.mp3'),
 Path('/Users/anshulchiranth/Desktop/Strike/Voice Experiments/Transcription/data/Fuquay_2-19_raw_audio/14-48-41.mp3'),
 Path('/Users/anshulchiranth/Desktop/Strike/Voice Experiments/Transcription/data/Fuquay_2-19_raw_audio/18-33-41.mp3'),
 Path('/Users/anshulchiranth/Desktop/Strike/Voice Experiments/Transcription/data/Fuquay_2-19_raw_audio/14-03-41.mp3'),
 Path('/Users/anshulchiranth/Desktop/Strike/Voice Experiments/Transcription/data/Fuquay_2-19_raw_audio/21-48-41.mp3'),
 Path('/Users/anshulchiranth/Desktop/Strike/Voice Experiments/Transcription/data/Fuquay_2-19_raw_audio/21-18-41.mp3')]

In [6]:
#Preprocess the raw_audio_files
#Pass through methods decode_audio(), preprocess_audio(), and save_audio()
#decode_audio() and preprocess_audio() combine to apply bandpass, AGC and limiter (in that order)
#save_audio() will save ground truths to "Processed Ground Truths"
from tqdm import tqdm

print(f"Found {len(raw_audio_files)} raw input files.")


processed_audio_samples = []

for i, file_path in enumerate(tqdm(raw_audio_files)):
    if file_path not in target_files:
        continue
    
    file_name = str(raw_audio_files[i])[len(str(raw_audio_dir))+1:]
    audio, sr = decode_audio(file_path)
    processed_audio_sample = preprocess_audio(audio, sr)
    output_path = save_audio(audio, sr, processed_audio_dir / f"Processed_{file_name}")
    sample_rate = sr
    processed_audio_samples.append(output_path)
    print(f"Saved file to {output_path}")

Found 96 raw input files.


 46%|████▌     | 44/96 [00:02<00:02, 21.71it/s]

Saved file to /Users/anshulchiranth/Desktop/Strike/Voice Experiments/Transcription/data/Processed Fuquay_2-19_audio/Processed_21-03-41.mp3
Saved file to /Users/anshulchiranth/Desktop/Strike/Voice Experiments/Transcription/data/Processed Fuquay_2-19_audio/Processed_14-48-41.mp3
Saved file to /Users/anshulchiranth/Desktop/Strike/Voice Experiments/Transcription/data/Processed Fuquay_2-19_audio/Processed_18-33-41.mp3


 49%|████▉     | 47/96 [00:07<00:10,  4.74it/s]

Saved file to /Users/anshulchiranth/Desktop/Strike/Voice Experiments/Transcription/data/Processed Fuquay_2-19_audio/Processed_14-03-41.mp3


 51%|█████     | 49/96 [00:09<00:12,  3.72it/s]

Saved file to /Users/anshulchiranth/Desktop/Strike/Voice Experiments/Transcription/data/Processed Fuquay_2-19_audio/Processed_21-48-41.mp3


100%|██████████| 96/96 [00:11<00:00,  8.34it/s]

Saved file to /Users/anshulchiranth/Desktop/Strike/Voice Experiments/Transcription/data/Processed Fuquay_2-19_audio/Processed_21-18-41.mp3


In [7]:
from utils import vad_split_pipeline, export_clips

In [8]:
for i, audio_path in enumerate(tqdm(processed_audio_samples)):
    audio, sr, speech_segments, merged_clips, clip_tensors = vad_split_pipeline(
    audio_path,
    target_sr=sample_rate,
    gap_s=7.0,
    threshold=0.5,
    min_speech_duration_ms=250,
    min_silence_duration_ms=500,
    pad_ms=200,
    )

    

    out_paths = export_clips(
    clip_tensors,
    sr,
    out_dir= data_dir / f"Clipped {audio_path.stem}",
    base_name="a",
    segments=merged_clips,  # so filenames include sample bounds
    )

100%|██████████| 6/6 [00:17<00:00,  2.89s/it]


### First Approach


In [9]:
#First goal: go from raw audio file to entire 15 minute transcript

In [10]:
from dotenv import load_dotenv
import os
from openai import OpenAI

env_path = Path("..") / ".env"

load_dotenv(env_path)

api_key = os.getenv("OPENAI_API_KEY")

client = OpenAI()

In [11]:
from utils import split_audio_equal_parts_gapless

In [12]:
import os
os.environ["PATH"] += os.pathsep + "/opt/homebrew/bin"

In [13]:
split_paths = split_audio_equal_parts_gapless(input_path=processed_audio_samples[0], output_dir = data_dir / "3_Minute_Clips", num_parts = 10)

ffmpeg version 7.1.1 Copyright (c) 2000-2025 the FFmpeg developers
  built with Apple clang version 17.0.0 (clang-1700.0.13.3)
  configuration: --prefix=/opt/homebrew/Cellar/ffmpeg/7.1.1_3 --enable-shared --enable-pthreads --enable-version3 --cc=clang --host-cflags= --host-ldflags='-Wl,-ld_classic' --enable-ffplay --enable-gnutls --enable-gpl --enable-libaom --enable-libaribb24 --enable-libbluray --enable-libdav1d --enable-libharfbuzz --enable-libjxl --enable-libmp3lame --enable-libopus --enable-librav1e --enable-librist --enable-librubberband --enable-libsnappy --enable-libsrt --enable-libssh --enable-libsvtav1 --enable-libtesseract --enable-libtheora --enable-libvidstab --enable-libvmaf --enable-libvorbis --enable-libvpx --enable-libwebp --enable-libx264 --enable-libx265 --enable-libxml2 --enable-libxvid --enable-lzma --enable-libfontconfig --enable-libfreetype --enable-frei0r --enable-libass --enable-libopencore-amrnb --enable-libopencore-amrwb --enable-libopenjpeg --enable-libspeex

In [14]:
split_paths

[PosixPath('/Users/anshulchiranth/Desktop/Strike/Voice Experiments/Transcription/data/3_Minute_Clips/Processed_21-03-41_part_000.wav'),
 PosixPath('/Users/anshulchiranth/Desktop/Strike/Voice Experiments/Transcription/data/3_Minute_Clips/Processed_21-03-41_part_001.wav'),
 PosixPath('/Users/anshulchiranth/Desktop/Strike/Voice Experiments/Transcription/data/3_Minute_Clips/Processed_21-03-41_part_002.wav'),
 PosixPath('/Users/anshulchiranth/Desktop/Strike/Voice Experiments/Transcription/data/3_Minute_Clips/Processed_21-03-41_part_003.wav'),
 PosixPath('/Users/anshulchiranth/Desktop/Strike/Voice Experiments/Transcription/data/3_Minute_Clips/Processed_21-03-41_part_004.wav'),
 PosixPath('/Users/anshulchiranth/Desktop/Strike/Voice Experiments/Transcription/data/3_Minute_Clips/Processed_21-03-41_part_005.wav'),
 PosixPath('/Users/anshulchiranth/Desktop/Strike/Voice Experiments/Transcription/data/3_Minute_Clips/Processed_21-03-41_part_006.wav'),
 PosixPath('/Users/anshulchiranth/Desktop/Strike

In [15]:
prompt_1 = """You are a performance reviewer assessing a Dairy Queen drive-thru operator's handling of an order. 
Create a transcript for the customer-operator interactions in this audio file. If you do not detect any speech,
that is ok, just return the phrase: "no speech detected."

Rules:
- Align the outputted transcript with the audio as close as possible. Do not try to clean up the spoken language, fix contractions, etc. for improved grammar.
- Never add in words that do not appear in the audio.
- Do not include any newlines in your response.
"""

In [16]:
transcriptions = []

for i, path in enumerate(split_paths):

    audio_file = open(path, "rb")
    transcription = client.audio.transcriptions.create(model = "gpt-4o-transcribe", file = audio_file, response_format = "text", prompt=prompt_1, 
                                                       language = "en")
    transcriptions.append(transcription)
    print(f"Done with trial: {i}")

INFO:httpx:HTTP Request: POST https://api.openai.com/v1/audio/transcriptions "HTTP/1.1 200 OK"


Done with trial: 0


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/audio/transcriptions "HTTP/1.1 200 OK"


Done with trial: 1


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/audio/transcriptions "HTTP/1.1 200 OK"


...

Done with trial: 7


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/audio/transcriptions "HTTP/1.1 200 OK"


Done with trial: 8


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/audio/transcriptions "HTTP/1.1 200 OK"


Done with trial: 9


In [17]:
#Step 2: For each of the 10 returned transcripts, identify how many transactions are present in a given transcript

prompt_2 = """You are a performance reviewer assessing a Dairy Queen drive-thru operator's handling of an order. You are given a transcript with transaction(s),
where a single transaction is a single customer-operator interaction. Your goal is to identify the number of distinct transactions in this transcript. Additionally,
you must identify whether each of these transactions is complete or incomplete.

For each transcript, return a JSON object with these fields:

{
    "num_transactions": "the number of distinct transactions you identified in this transcript",
    "transcriptions": "a list where the first entry is the transcript corresponding to the first recognized transaction, the second entry is the transcript corresponding to the second recognized transaction, etc.",
    "transaction_statuses": "a list where the first entry is the status (complete/incomplete) of the first recognized transaction, the second entry is the status of the second recognized transaction, etc."
}

Rules:
- Preserve the exact wording of the given transcript. The "transcriptions" list is just a split up version of the given transcript
- "num_transactions" = len("transcriptions") = len("transaction_statuses"). This should always be the case and never be violated
- Transactions generally start with phrases like "Welcome to Dairy Queen"
- Transactions generally end with phrases like "thank you so much" or "your total will be at the window"
- An empty given transcript is possible. These generally just have the text: "prompt" or "no speech detected." In these instances, "num_transactions" = 0, "transcriptions" = [] and "transaction_statuses" = []
"""

In [18]:
responses = []

for i, transcript in enumerate(transcriptions):
    final_prompt2 = f"""{prompt_2}

    Process this transcript and return the JSON result:

    ---
    {transcript}
    ---

    Return ONLY the JSON object, no other text."""

    messages = [{"role": "user", "content": final_prompt2}]
    kwargs = {
                "model": "gpt-4o",
                "messages": messages,
                "temperature": 0,
            }
    response = client.chat.completions.create(**kwargs)
    raw_content = response.choices[0].message.content
    cleaned = raw_content.strip().removeprefix("```json").removesuffix("```").strip()
    data = json.loads(cleaned)

    responses.append(data)
    print(f"Done with trial {i}")

INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


Done with trial 0


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


Done with trial 1


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


...

Done with trial 7


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


Done with trial 8


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


Done with trial 9


In [19]:
#Find timestamps using stable-whisper
import stable_whisper
from utils import best_segment_window_full, clip_audio_gapless_with_padding, best_segment_window_anchors

In [29]:
clip_dir = data_dir / "3_Minute_Clips"

model = stable_whisper.load_model("base")


for i, (response, file_path) in enumerate(zip(responses, split_paths)):
    print(file_path)    
    whisper_transcript = model.transcribe(str(file_path), language = "en")
    for real_transcript in response["transcriptions"]:
        window = best_segment_window_full(whisper_transcript, real_transcript)
        if window is None:
            # Skip: empty transcript or no match window found
            continue
    
        t0, t1, score, i, j, matched_text = window
        out_path = clip_audio_gapless_with_padding(
            file_path,
            data_dir / "Whisper_Clips",
            t0,
            t1,
        )
        print(out_path)

/Users/anshulchiranth/Desktop/Strike/Voice Experiments/Transcription/data/3_Minute_Clips/Processed_21-03-41_part_000.wav


/opt/anaconda3/envs/voice_id_env/lib/python3.11/site-packages/stable_whisper/whisper_word_level/original_whisper.py:249: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")
Transcribe: 100%|██████████| 89.98/89.98 [00:03<00:00, 24.66sec/s]
ffmpeg version 7.1.1 Copyright (c) 2000-2025 the FFmpeg developers
  built with Apple clang version 17.0.0 (clang-1700.0.13.3)
  configuration: --prefix=/opt/homebrew/Cellar/ffmpeg/7.1.1_3 --enable-shared --enable-pthreads --enable-version3 --cc=clang --host-cflags= --host-ldflags='-Wl,-ld_classic' --enable-ffplay --enable-gnutls --enable-gpl --enable-libaom --enable-libaribb24 --enable-libbluray --enable-libdav1d --enable-libharfbuzz --enable-libjxl --enable-libmp3lame --enable-libopus --enable-librav1e --enable-librist --enable-librubberband --enable-libsnappy --enable-libsrt --enable-libssh --enable-libsvtav1 --enable-libtesseract --enable-libtheora --enable-libvidstab -

/Users/anshulchiranth/Desktop/Strike/Voice Experiments/Transcription/data/Whisper_Clips/Processed_21-03-41_part_000_clip_64.68s_89.98s.wav
/Users/anshulchiranth/Desktop/Strike/Voice Experiments/Transcription/data/3_Minute_Clips/Processed_21-03-41_part_001.wav


Transcribe: 100%|██████████| 89.98/89.98 [00:09<00:00,  9.09sec/s]
ffmpeg version 7.1.1 Copyright (c) 2000-2025 the FFmpeg developers
  built with Apple clang version 17.0.0 (clang-1700.0.13.3)
  configuration: --prefix=/opt/homebrew/Cellar/ffmpeg/7.1.1_3 --enable-shared --enable-pthreads --enable-version3 --cc=clang --host-cflags= --host-ldflags='-Wl,-ld_classic' --enable-ffplay --enable-gnutls --enable-gpl --enable-libaom --enable-libaribb24 --enable-libbluray --enable-libdav1d --enable-libharfbuzz --enable-libjxl --enable-libmp3lame --enable-libopus --enable-librav1e --enable-librist --enable-librubberband --enable-libsnappy --enable-libsrt --enable-libssh --enable-libsvtav1 --enable-libtesseract --enable-libtheora --enable-libvidstab --enable-libvmaf --enable-libvorbis --enable-libvpx --enable-libwebp --enable-libx264 --enable-libx265 --enable-libxml2 --enable-libxvid --enable-lzma --enable-libfontconfig --enable-libfreetype --enable-frei0r --enable-libass --enable-libopencore-amrn

/Users/anshulchiranth/Desktop/Strike/Voice Experiments/Transcription/data/Whisper_Clips/Processed_21-03-41_part_001_clip_3.56s_85.24s.wav
/Users/anshulchiranth/Desktop/Strike/Voice Experiments/Transcription/data/3_Minute_Clips/Processed_21-03-41_part_002.wav


...

Transcribe: 100%|██████████| 89.98/89.98 [00:04<00:00, 18.09sec/s]
ffmpeg version 7.1.1 Copyright (c) 2000-2025 the FFmpeg developers
  built with Apple clang version 17.0.0 (clang-1700.0.13.3)
  configuration: --prefix=/opt/homebrew/Cellar/ffmpeg/7.1.1_3 --enable-shared --enable-pthreads --enable-version3 --cc=clang --host-cflags= --host-ldflags='-Wl,-ld_classic' --enable-ffplay --enable-gnutls --enable-gpl --enable-libaom --enable-libaribb24 --enable-libbluray --enable-libdav1d --enable-libharfbuzz --enable-libjxl --enable-libmp3lame --enable-libopus --enable-librav1e --enable-librist --enable-librubberband --enable-libsnappy --enable-libsrt --enable-libssh --enable-libsvtav1 --enable-libtesseract --enable-libtheora --enable-libvidstab --enable-libvmaf --enable-libvorbis --enable-libvpx --enable-libwebp --enable-libx264 --enable-libx265 --enable-libxml2 --enable-libxvid --enable-lzma --enable-libfontconfig --enable-libfreetype --enable-frei0r --enable-libass --enable-libopencore-amrn

/Users/anshulchiranth/Desktop/Strike/Voice Experiments/Transcription/data/Whisper_Clips/Processed_21-03-41_part_007_clip_83.24s_89.14s.wav
/Users/anshulchiranth/Desktop/Strike/Voice Experiments/Transcription/data/3_Minute_Clips/Processed_21-03-41_part_008.wav


Transcribe: 100%|██████████| 89.98/89.98 [00:03<00:00, 23.67sec/s]
/opt/anaconda3/envs/voice_id_env/lib/python3.11/site-packages/stable_whisper/whisper_word_level/original_whisper.py:249: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


/Users/anshulchiranth/Desktop/Strike/Voice Experiments/Transcription/data/3_Minute_Clips/Processed_21-03-41_part_009.wav


Transcribe: 100%|██████████| 89.98/89.98 [00:05<00:00, 16.27sec/s]
/opt/anaconda3/envs/voice_id_env/lib/python3.11/site-packages/stable_whisper/result.py:2050: UserWarning: Cannot clamp due to missing/no word-timestamps
  warnings.warn('Cannot clamp due to missing/no word-timestamps')
/opt/anaconda3/envs/voice_id_env/lib/python3.11/site-packages/stable_whisper/whisper_word_level/original_whisper.py:706: UserWarning: Failed to transcribe audio. Result contains no text. 
  warnings.warn(f'Failed to {task} audio. Result contains no text. ')


In [21]:
clip_dir = data_dir / "3_Minute_Clips"

model = stable_whisper.load_model("base")


for i, (response, file_path) in enumerate(zip(responses, split_paths)):
    print(file_path)    
    whisper_transcript = model.transcribe(str(file_path), language = "en")
    for real_transcript in response["transcriptions"]:

        window_and_hits = best_segment_window_anchors(whisper_transcript, real_transcript)
        if window_and_hits is None:
            # Skip: empty transcript or no match window found
            continue
            print("No anchor window found; skipping.")
    
        (t0, t1, score, i, j, matched_text), hits = window_and_hits
        print("Window:", t0, t1, "score:", score, "segments:", i, j)
        for h in hits:
            print("  hit:", h["score"], "anchor:", h["anchor"])
        out_path = clip_audio_gapless_with_padding(
            file_path,
            data_dir / "Whisper_Anchor_Clips",
            t0,
            t1,
        )
        print(out_path)

/opt/anaconda3/envs/voice_id_env/lib/python3.11/site-packages/stable_whisper/whisper_word_level/original_whisper.py:249: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


/Users/anshulchiranth/Desktop/Strike/Voice Experiments/Transcription/data/3_Minute_Clips/Processed_21-03-41_part_000.wav


Transcribe: 100%|██████████| 89.98/89.98 [00:03<00:00, 26.86sec/s]
ffmpeg version 7.1.1 Copyright (c) 2000-2025 the FFmpeg developers
  built with Apple clang version 17.0.0 (clang-1700.0.13.3)
  configuration: --prefix=/opt/homebrew/Cellar/ffmpeg/7.1.1_3 --enable-shared --enable-pthreads --enable-version3 --cc=clang --host-cflags= --host-ldflags='-Wl,-ld_classic' --enable-ffplay --enable-gnutls --enable-gpl --enable-libaom --enable-libaribb24 --enable-libbluray --enable-libdav1d --enable-libharfbuzz --enable-libjxl --enable-libmp3lame --enable-libopus --enable-librav1e --enable-librist --enable-librubberband --enable-libsnappy --enable-libsrt --enable-libssh --enable-libsvtav1 --enable-libtesseract --enable-libtheora --enable-libvidstab --enable-libvmaf --enable-libvorbis --enable-libvpx --enable-libwebp --enable-libx264 --enable-libx265 --enable-libxml2 --enable-libxvid --enable-lzma --enable-libfontconfig --enable-libfreetype --enable-frei0r --enable-libass --enable-libopencore-amrn

Window: 66.68 83.02 score: 95.00721201999451 segments: 0 9
  hit: 91.66666666666666 anchor: welcome to dairy queen will you be
  hit: 94.44444444444444 anchor: you be using your mobile rewards today
  hit: 94.11764705882352 anchor: rewards today no alright how can i
  hit: 91.30434782608697 anchor: can i get you started can i
  hit: 88.52459016393442 anchor: can i have a medium choco brownie
  hit: 100.0 anchor: choco brownie extreme blizzard alright anything else
  hit: 100.0 anchor: anything else yes a small oreo blizzard
  hit: 100.0 anchor: oreo blizzard alright small oreo and the
/Users/anshulchiranth/Desktop/Strike/Voice Experiments/Transcription/data/Whisper_Anchor_Clips/Processed_21-03-41_part_000_clip_64.68s_85.02s.wav
/Users/anshulchiranth/Desktop/Strike/Voice Experiments/Transcription/data/3_Minute_Clips/Processed_21-03-41_part_001.wav


Transcribe: 100%|██████████| 89.98/89.98 [00:10<00:00,  8.39sec/s]
ffmpeg version 7.1.1 Copyright (c) 2000-2025 the FFmpeg developers
  built with Apple clang version 17.0.0 (clang-1700.0.13.3)
  configuration: --prefix=/opt/homebrew/Cellar/ffmpeg/7.1.1_3 --enable-shared --enable-pthreads --enable-version3 --cc=clang --host-cflags= --host-ldflags='-Wl,-ld_classic' --enable-ffplay --enable-gnutls --enable-gpl --enable-libaom --enable-libaribb24 --enable-libbluray --enable-libdav1d --enable-libharfbuzz --enable-libjxl --enable-libmp3lame --enable-libopus --enable-librav1e --enable-librist --enable-librubberband --enable-libsnappy --enable-libsrt --enable-libssh --enable-libsvtav1 --enable-libtesseract --enable-libtheora --enable-libvidstab --enable-libvmaf --enable-libvorbis --enable-libvpx --enable-libwebp --enable-libx264 --enable-libx265 --enable-libxml2 --enable-libxvid --enable-lzma --enable-libfontconfig --enable-libfreetype --enable-frei0r --enable-libass --enable-libopencore-amrn

...

Transcribe: 100%|██████████| 89.98/89.98 [00:04<00:00, 18.41sec/s]
/opt/anaconda3/envs/voice_id_env/lib/python3.11/site-packages/stable_whisper/whisper_word_level/original_whisper.py:249: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


/Users/anshulchiranth/Desktop/Strike/Voice Experiments/Transcription/data/3_Minute_Clips/Processed_21-03-41_part_008.wav


Transcribe: 100%|██████████| 89.98/89.98 [00:01<00:00, 48.94sec/s]
ffmpeg version 7.1.1 Copyright (c) 2000-2025 the FFmpeg developers
  built with Apple clang version 17.0.0 (clang-1700.0.13.3)
  configuration: --prefix=/opt/homebrew/Cellar/ffmpeg/7.1.1_3 --enable-shared --enable-pthreads --enable-version3 --cc=clang --host-cflags= --host-ldflags='-Wl,-ld_classic' --enable-ffplay --enable-gnutls --enable-gpl --enable-libaom --enable-libaribb24 --enable-libbluray --enable-libdav1d --enable-libharfbuzz --enable-libjxl --enable-libmp3lame --enable-libopus --enable-librav1e --enable-librist --enable-librubberband --enable-libsnappy --enable-libsrt --enable-libssh --enable-libsvtav1 --enable-libtesseract --enable-libtheora --enable-libvidstab --enable-libvmaf --enable-libvorbis --enable-libvpx --enable-libwebp --enable-libx264 --enable-libx265 --enable-libxml2 --enable-libxvid --enable-lzma --enable-libfontconfig --enable-libfreetype --enable-frei0r --enable-libass --enable-libopencore-amrn

Window: 2.47 5.64 score: 88.46153846153845 segments: 0 0
  hit: 76.92307692307692 anchor: ll be it okay it doesn t
  hit: 100.0 anchor: doesn t be 540 at the window
/Users/anshulchiranth/Desktop/Strike/Voice Experiments/Transcription/data/Whisper_Anchor_Clips/Processed_21-03-41_part_008_clip_0.47s_7.64s.wav
/Users/anshulchiranth/Desktop/Strike/Voice Experiments/Transcription/data/3_Minute_Clips/Processed_21-03-41_part_009.wav


Transcribe: 100%|██████████| 89.98/89.98 [00:05<00:00, 16.45sec/s]
/opt/anaconda3/envs/voice_id_env/lib/python3.11/site-packages/stable_whisper/result.py:2050: UserWarning: Cannot clamp due to missing/no word-timestamps
  warnings.warn('Cannot clamp due to missing/no word-timestamps')
/opt/anaconda3/envs/voice_id_env/lib/python3.11/site-packages/stable_whisper/whisper_word_level/original_whisper.py:706: UserWarning: Failed to transcribe audio. Result contains no text. 
  warnings.warn(f'Failed to {task} audio. Result contains no text. ')


In [22]:
tscript = model.transcribe(str("/Users/anshulchiranth/Desktop/Strike/Voice Experiments/Transcription/data/3_Minute_Clips/Processed_21-03-41_part_008.wav"), language = "en")

/opt/anaconda3/envs/voice_id_env/lib/python3.11/site-packages/stable_whisper/whisper_word_level/original_whisper.py:249: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")
Transcribe: 100%|██████████| 89.98/89.98 [00:02<00:00, 42.72sec/s]


In [23]:
tscript.text

" Okay, it doesn't be 540 at the window."

In [24]:
tscript.duration

np.float64(3.17)

In [25]:
tscript.segments

[Segment(start=2.47, end=5.64, text=" Okay, it doesn't be 540 at the window.")]

In [28]:
responses

[{'num_transactions': 1,
  'transcriptions': ['Welcome to Dairy Queen, will you be using your mobile rewards today? No. Alright, how can I get you started? Can I have a medium choco brownie extreme blizzard? Alright, anything else? Yes, a small Oreo blizzard. Alright, small Oreo. And the last thing is a small blizzard with brownie and cookie dough.'],
  'transaction_statuses': ['incomplete']},
 {'num_transactions': 1,
  'transcriptions': ["Is it a small Oreo incognito? Brownie incognito. Are you good with the chocolate fudge that comes with that? No thank you. All right. Will that be all? Yes. All right, there's gonna be 1782 at the window. Thank you. No problem."],
  'transaction_statuses': ['complete']},
 {'num_transactions': 1,
  'transcriptions': ["Welcome to Dairy Queen, how can I start your order? Hi, um, can I do... Can I get, um, two mini red velvet cake blizzards with Oreos? All right, anything else? Um, and then can I do, um, a mini of the chocolate brownie extreme blizzard? 

In [18]:
#Step 3: Combine transcripts based upon their status

In [19]:
tx_transcripts = []
last_transaction_status = "complete"


for response in responses:
    transcriptions = response["transcriptions"]
    transaction_statuses = response["transaction_statuses"]
    for transcript, status in zip(transcriptions, transaction_statuses):
        #Start a new transaction
        if last_transaction_status == "complete":
            tx_transcripts.append(transcript)
            last_transaction_status = status
        elif last_transaction_status == "incomplete":
            existing = tx_transcripts[-1]
            tx_transcripts[-1] = existing + transcript
            last_transaction_status = status


In [38]:
tx_transcripts

["Welcome to Dairy Queen. Will you be using your mobile rewards today? No. All right, how can I get you started? Can I have a medium Choco Brownie Extreme Blizzard? All right, anything else? Yes, a small Oreo Blizzard. All right, small Oreo. And the last thing is a small Blizzard with brownie and cookie dough.Is there a small Oreo and cookie dough? Brownie and cookie dough. Are you good with the chocolate fudge that comes with that? No, thank you. Will that be all? Yes. Alright, there's only $17.82 at the window. Thank you. No problem.",
 "Welcome to Dairy Queen, how can I start your order? Hi, um, can I do... Can I get, um, two mini red velvet cake blizzards with Oreos? Alright, anything else? Um, and then can I do a mini of the chocolate brownie extreme blizzard? Alright. And that's it. Did you want a drink to go with this order? No. Alright, that would be $15.53 at the window. Thank you. Yeah, no problem.",
 "Welcome to Dairy Queen, we'll be using your mobile rewards today. Not toda

In [39]:
responses

[{'num_transactions': 1,
  'transcriptions': ['Welcome to Dairy Queen. Will you be using your mobile rewards today? No. All right, how can I get you started? Can I have a medium Choco Brownie Extreme Blizzard? All right, anything else? Yes, a small Oreo Blizzard. All right, small Oreo. And the last thing is a small Blizzard with brownie and cookie dough.'],
  'transaction_statuses': ['incomplete']},
 {'num_transactions': 1,
  'transcriptions': ["Is there a small Oreo and cookie dough? Brownie and cookie dough. Are you good with the chocolate fudge that comes with that? No, thank you. Will that be all? Yes. Alright, there's only $17.82 at the window. Thank you. No problem."],
  'transaction_statuses': ['complete']},
 {'num_transactions': 1,
  'transcriptions': ["Welcome to Dairy Queen, how can I start your order? Hi, um, can I do... Can I get, um, two mini red velvet cake blizzards with Oreos? Alright, anything else? Um, and then can I do a mini of the chocolate brownie extreme blizzard

In [20]:
#Step 4: Obtain timestamps given each transaction's transcript

In [21]:
from utils import get_timestamps_option_a_whisperx

In [22]:
import stable_whisper

In [23]:
model = stable_whisper.load_model("base")
result = model.align(str(processed_audio_samples[0]), tx_transcripts[0], language = "en")

Align: 100%|██████████| 899.93/899.93 [00:06<00:00, 143.41sec/s]
Adjustment: 100%|██████████| 114.35/114.35 [00:00<00:00, 208620.56sec/s]
/opt/anaconda3/envs/voice_id_env/lib/python3.11/site-packages/stable_whisper/alignment.py:211: UserWarning: 6/18 segments failed to align.
  result = aligner.align(audio, text)


In [24]:
result

In [25]:
result.to_tsv('audio2.tsv')

Saved: /Users/anshulchiranth/Desktop/Strike/Voice Experiments/Transcription/audio2.tsv


In [26]:
result.to_srt_vtt("audio2.srt")

Saved: /Users/anshulchiranth/Desktop/Strike/Voice Experiments/Transcription/audio2.srt


In [27]:
processed_audio_samples[0]

PosixPath('/Users/anshulchiranth/Desktop/Strike/Voice Experiments/Transcription/data/Processed Fuquay_2-19_audio/Processed_21-03-41.mp3')

In [28]:
result.to_srt_vtt('audio2.vtt')

Saved: /Users/anshulchiranth/Desktop/Strike/Voice Experiments/Transcription/audio2.vtt


In [29]:
result.to_ass('audio2.ass')

Saved: /Users/anshulchiranth/Desktop/Strike/Voice Experiments/Transcription/audio2.ass


In [30]:
matches = model.locate(str(processed_audio_samples[0]), tx_transcripts[0], language = "en")
for match in matches:
    print(math.to_display_str())

Locate: 100%|██████████| 899.84/899.84 [00:34<00:00, 26.24sec/s]


In [31]:
matches

[]

In [33]:
tx_transcripts

["Welcome to Dairy Queen. Will you be using your mobile rewards today? No. All right, how can I get you started? Can I have a medium Choco Brownie Extreme Blizzard? All right, anything else? Yes, a small Oreo Blizzard. All right, small Oreo. And the last thing is a small Blizzard with brownie and cookie dough.Is there a small Oreo and cookie dough? Brownie and cookie dough. Are you good with the chocolate fudge that comes with that? No, thank you. Will that be all? Yes. Alright, there's only $17.82 at the window. Thank you. No problem.",
 "Welcome to Dairy Queen, how can I start your order? Hi, um, can I do... Can I get, um, two mini red velvet cake blizzards with Oreos? Alright, anything else? Um, and then can I do a mini of the chocolate brownie extreme blizzard? Alright. And that's it. Did you want a drink to go with this order? No. Alright, that would be $15.53 at the window. Thank you. Yeah, no problem.",
 "Welcome to Dairy Queen, we'll be using your mobile rewards today. Not toda

In [35]:
result = model.transcribe(str(processed_audio_samples[0]), language = "en")
matches = result.find(tx_transcripts[0])

for match in matches:
  print(f'match: {match.text_match}\n'
        f'text: {match.text}\n'
        f'start: {match.start}\n'
        f'end: {match.end}\n')

/opt/anaconda3/envs/voice_id_env/lib/python3.11/site-packages/stable_whisper/whisper_word_level/original_whisper.py:249: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")
Transcribe: 100%|██████████| 899.84/899.84 [00:58<00:00, 15.48sec/s]


In [36]:
matches